In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error

from v2_eval import MODEL_FEATURES, ABLATION_CANDIDATES, iter_block_splits, within_job_spearman

feat = pd.read_parquet("../data/processed/v2_features.parquet")
FLAG_FEATURES = ["has_career_objective", "has_certifications", "has_languages", "has_extra_curricular"]

print("Model features:", MODEL_FEATURES)
print(feat.shape)

Model features: ['semantic_similarity', 'objective_responsibilities_similarity', 'education_similarity', 'skills_soft_coverage', 'experience_gap', 'candidate_experience_years']
(9544, 19)


In [2]:
def evaluate_baseline(feat, score_fn, name):
    rows = []
    for rf, jf, _, te in iter_block_splits(feat["resume_fold"], feat["job_fold"]):
        y_true = feat.loc[te, "matched_score"]
        y_score = score_fn(feat.loc[te])
        rows.append(within_job_spearman(y_true, y_score, feat.loc[te, "job_id"]))
    return pd.Series(rows, name=name)

baseline_best_feature = evaluate_baseline(
    feat, lambda df: df["semantic_similarity"], "baseline_semantic_only"
)
baseline_soft_skills = evaluate_baseline(
    feat, lambda df: df["skills_soft_coverage"], "baseline_skills_soft_only"
)
baseline_mean = evaluate_baseline(
    feat,
    lambda df: df[["semantic_similarity", "skills_soft_coverage", "education_similarity"]].mean(axis=1, skipna=True),
    "baseline_simple_mean",
)

baselines = pd.concat([baseline_best_feature, baseline_soft_skills, baseline_mean], axis=1)
print(baselines.mean().round(3))
print(baselines.std().round(3))

baseline_semantic_only       0.297
baseline_skills_soft_only    0.300
baseline_simple_mean         0.370
dtype: float64
baseline_semantic_only       0.092
baseline_skills_soft_only    0.108
baseline_simple_mean         0.093
dtype: float64


In [3]:
def build_estimator(model_name):
    model = {
        "ridge": Ridge(alpha=1.0, random_state=42),
        "random_forest": RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=3, random_state=42, n_jobs=-1),
        "gradient_boosting": GradientBoostingRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, min_samples_leaf=3, random_state=42),
    }[model_name]
    # Uniform imputation for all 3 models: a fair comparison,
    # et GradientBoostingRegressor ne supporte pas nativement les NaN.
    return Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("model", model),
    ])


def evaluate_config(feat, feature_cols, model_name):
    rows = []
    for rf, jf, tr, te in iter_block_splits(feat["resume_fold"], feat["job_fold"]):
        X_train, y_train = feat.loc[tr, feature_cols], feat.loc[tr, "matched_score"]
        X_test, y_test = feat.loc[te, feature_cols], feat.loc[te, "matched_score"]

        est = build_estimator(model_name)
        est.fit(X_train, y_train)
        y_pred = est.predict(X_test)

        rows.append({
            "resume_fold": rf, "job_fold": jf,
            "within_job_spearman": within_job_spearman(y_test, y_pred, feat.loc[te, "job_id"]),
            "r2": r2_score(y_test, y_pred),
            "mae": mean_absolute_error(y_test, y_pred),
        })
    return pd.DataFrame(rows)

In [4]:
results = {}
for model_name in ["ridge", "random_forest", "gradient_boosting"]:
    print(f"{model_name}...")
    results[model_name] = evaluate_config(feat, MODEL_FEATURES, model_name)

summary = pd.DataFrame({
    name: {
        "spearman_mean": df["within_job_spearman"].mean(),
        "spearman_std": df["within_job_spearman"].std(),
        "r2_mean": df["r2"].mean(),
        "mae_mean": df["mae"].mean(),
    }
    for name, df in results.items()
}).T
print(summary.round(3))

ridge...
random_forest...
gradient_boosting...
                   spearman_mean  spearman_std  r2_mean  mae_mean
ridge                      0.320         0.076    0.013     0.129
random_forest              0.296         0.089   -0.012     0.128
gradient_boosting          0.304         0.093    0.006     0.128


In [5]:
best_model = summary["spearman_mean"].idxmax()
print(f"Best model : {best_model}")

ablation_results = {}

ablation_results["full"] = results[best_model]["within_job_spearman"]

ablation_results["+ skills_exact"] = evaluate_config(
    feat, MODEL_FEATURES + ABLATION_CANDIDATES, best_model
)["within_job_spearman"]

ablation_results["- candidate_experience_years"] = evaluate_config(
    feat, [c for c in MODEL_FEATURES if c != "candidate_experience_years"], best_model
)["within_job_spearman"]

ablation_results["+ flags has_*"] = evaluate_config(
    feat, MODEL_FEATURES + FLAG_FEATURES, best_model
)["within_job_spearman"]

ablation_summary = pd.DataFrame(ablation_results).agg(["mean", "std"]).T
print(ablation_summary.round(3))

Best model : ridge
                               mean    std
full                          0.320  0.076
+ skills_exact                0.318  0.071
- candidate_experience_years  0.331  0.090
+ flags has_*                 0.328  0.075


In [6]:
final = pd.DataFrame({
    "baseline_semantic_only": baselines["baseline_semantic_only"],
    "baseline_skills_soft_only": baselines["baseline_skills_soft_only"],
    "baseline_simple_mean": baselines["baseline_simple_mean"],
    **{f"model_{name}": df["within_job_spearman"] for name, df in results.items()},
}).agg(["mean", "std"]).T.sort_values("mean", ascending=False)

print(final.round(3))

                            mean    std
baseline_simple_mean       0.370  0.093
model_ridge                0.320  0.076
model_gradient_boosting    0.304  0.093
baseline_skills_soft_only  0.300  0.108
baseline_semantic_only     0.297  0.092
model_random_forest        0.296  0.089


In [7]:
summary.to_csv("../data/processed/v2_model_comparison.csv")
ablation_summary.to_csv("../data/processed/v2_ablation_comparison.csv")
print("Results saved.")

Results saved.


In [8]:
def evaluate_with_keys(feat, score_fn, trained=False, feature_cols=None, model_name=None):
    rows = []
    for rf, jf, tr, te in iter_block_splits(feat["resume_fold"], feat["job_fold"]):
        y_test = feat.loc[te, "matched_score"]
        if trained:
            est = build_estimator(model_name)
            est.fit(feat.loc[tr, feature_cols], feat.loc[tr, "matched_score"])
            y_score = est.predict(feat.loc[te, feature_cols])
        else:
            y_score = score_fn(feat.loc[te])
        rows.append({
            "resume_fold": rf, "job_fold": jf,
            "spearman": within_job_spearman(y_test, y_score, feat.loc[te, "job_id"]),
            "pred_std": float(np.std(y_score)),
        })
    return pd.DataFrame(rows).set_index(["resume_fold", "job_fold"])


baseline_df = evaluate_with_keys(
    feat, lambda df: df[["semantic_similarity", "skills_soft_coverage", "education_similarity"]].mean(axis=1, skipna=True)
)
ridge_df = evaluate_with_keys(feat, None, trained=True, feature_cols=MODEL_FEATURES, model_name="ridge")

paired = baseline_df[["spearman"]].join(ridge_df[["spearman"]], lsuffix="_baseline", rsuffix="_ridge")
paired["diff"] = paired["spearman_baseline"] - paired["spearman_ridge"]

print(paired.round(3))
print(f"\nMean difference : {paired['diff'].mean():.3f}")
print(f"Baseline gagne sur {(paired['diff'] > 0).sum()}/25 splits")

from scipy.stats import wilcoxon
stat, p = wilcoxon(paired["spearman_baseline"], paired["spearman_ridge"])
print(f"Paired Wilcoxon test : p = {p:.3f}")

print(f"\nStandard deviation of Ridge predictions per split :")
print(ridge_df["pred_std"].describe().round(4))

                      spearman_baseline  spearman_ridge   diff
resume_fold job_fold                                          
0           0                     0.167           0.215 -0.049
            1                     0.399           0.350  0.049
            2                     0.283           0.304 -0.020
            3                     0.366           0.421 -0.054
            4                     0.480           0.376  0.104
1           0                     0.281           0.248  0.033
            1                     0.406           0.362  0.044
            2                     0.347           0.259  0.088
            3                     0.374           0.439 -0.065
            4                     0.467           0.300  0.167
2           0                     0.292           0.304 -0.012
            1                     0.516           0.461  0.055
            2                     0.367           0.264  0.103
            3                     0.383           0.375

In [9]:
from sklearn.linear_model import Ridge

def evaluate_ridge_alpha(feat, feature_cols, alpha):
    rows = []
    for rf, jf, tr, te in iter_block_splits(feat["resume_fold"], feat["job_fold"]):
        est = Pipeline([
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("model", Ridge(alpha=alpha, random_state=42)),
        ])
        est.fit(feat.loc[tr, feature_cols], feat.loc[tr, "matched_score"])
        y_pred = est.predict(feat.loc[te, feature_cols])
        rows.append({
            "spearman": within_job_spearman(feat.loc[te, "matched_score"], y_pred, feat.loc[te, "job_id"]),
            "pred_std": float(np.std(y_pred)),
        })
    return pd.DataFrame(rows)

for alpha in [1.0, 0.1, 0.01, 0.001]:
    res = evaluate_ridge_alpha(feat, MODEL_FEATURES, alpha)
    print(f"alpha={alpha:<6} spearman={res['spearman'].mean():.3f}  pred_std={res['pred_std'].mean():.4f}")

alpha=1.0    spearman=0.320  pred_std=0.0639
alpha=0.1    spearman=0.319  pred_std=0.0644
alpha=0.01   spearman=0.319  pred_std=0.0645
alpha=0.001  spearman=0.319  pred_std=0.0645


In [10]:
corr = feat[MODEL_FEATURES].corr(method="spearman")
print(corr.round(2))

                                       semantic_similarity  \
semantic_similarity                                   1.00   
objective_responsibilities_similarity                 0.72   
education_similarity                                  0.14   
skills_soft_coverage                                  0.56   
experience_gap                                        0.23   
candidate_experience_years                            0.21   

                                       objective_responsibilities_similarity  \
semantic_similarity                                                     0.72   
objective_responsibilities_similarity                                   1.00   
education_similarity                                                    0.13   
skills_soft_coverage                                                    0.30   
experience_gap                                                         -0.05   
candidate_experience_years                                             -0.01   

    

In [11]:
est = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("model", Ridge(alpha=1.0, random_state=42)),
])
est.fit(feat[MODEL_FEATURES], feat["matched_score"])

imputer = est.named_steps["impute"]
feature_names_out = list(imputer.get_feature_names_out(MODEL_FEATURES))
coefs = pd.Series(est.named_steps["model"].coef_, index=feature_names_out)
print(coefs.sort_values())
print(f"\nintercept : {est.named_steps['model'].intercept_:.3f}")
print(f"Actual matched_score : mean={feat['matched_score'].mean():.3f}, std={feat['matched_score'].std():.3f}")

missingindicator_candidate_experience_years              -0.064825
missingindicator_education_similarity                    -0.034755
missingindicator_objective_responsibilities_similarity   -0.009856
candidate_experience_years                               -0.000009
experience_gap                                            0.001207
skills_soft_coverage                                      0.023548
missingindicator_experience_gap                           0.026616
missingindicator_skills_soft_coverage                     0.055665
education_similarity                                      0.149798
semantic_similarity                                       0.265494
objective_responsibilities_similarity                     0.299081
dtype: float64

intercept : 0.445
Actual matched_score : mean=0.661, std=0.167


In [12]:
baseline_score = feat[["semantic_similarity", "skills_soft_coverage", "education_similarity"]].mean(axis=1, skipna=True)
ridge_score = est.predict(feat[MODEL_FEATURES])

print(f"Correlation between Ridge predictions and the simple average: {pd.Series(ridge_score).corr(pd.Series(baseline_score)):.3f}")
print(f"std baseline : {baseline_score.std():.4f}  |  std ridge : {pd.Series(ridge_score).std():.4f}")

Correlation between Ridge predictions and the simple average: 0.798
std baseline : 0.0905  |  std ridge : 0.0649


In [13]:
MODEL_FEATURES_V2 = [f for f in MODEL_FEATURES if f != "candidate_experience_years"]

for name, cols in [
    ("without candidate_experience_years", MODEL_FEATURES_V2),
    ("also without objective_responsibilities_similarity", [c for c in MODEL_FEATURES_V2 if c != "objective_responsibilities_similarity"]),
]:
    res = evaluate_config(feat, cols, "ridge")
    print(f"{name:55} spearman={res['within_job_spearman'].mean():.3f}  std={res['within_job_spearman'].std():.3f}")

without candidate_experience_years                         spearman=0.331  std=0.090
also without objective_responsibilities_similarity        spearman=0.342  std=0.091


In [14]:
est = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("model", Ridge(alpha=1.0, random_state=42)),
])
clean_features = ["semantic_similarity", "education_similarity", "skills_soft_coverage", "experience_gap"]
est.fit(feat[clean_features], feat["matched_score"])
coefs = pd.Series(est.named_steps["model"].coef_[:len(clean_features)], index=clean_features)
print(coefs.sort_values())

res = evaluate_config(feat, clean_features, "ridge")
print(f"\nspearman={res['within_job_spearman'].mean():.3f}  std={res['within_job_spearman'].std():.3f}")

baseline4 = evaluate_baseline(
    feat, lambda df: df[clean_features].mean(axis=1, skipna=True), "baseline4"
)
print(f"baseline (mean of 4) spearman={baseline4.mean():.3f}")

experience_gap          0.000853
skills_soft_coverage    0.028204
education_similarity    0.150231
semantic_similarity     0.370974
dtype: float64

spearman=0.342  std=0.091
baseline (mean of 4) spearman=0.248


In [15]:
best_three = ["semantic_similarity", "education_similarity", "skills_soft_coverage"]

res = evaluate_config(feat, best_three, "ridge")
print(f"Ridge (3 best features)   spearman={res['within_job_spearman'].mean():.3f}  std={res['within_job_spearman'].std():.3f}")

baseline3 = evaluate_baseline(
    feat, lambda df: df[best_three].mean(axis=1, skipna=True), "baseline3"
)
print(f"baseline (mean of 3)    spearman={baseline3.mean():.3f}")

Ridge (3 best features)   spearman=0.346  std=0.094
baseline (mean of 3)    spearman=0.370


In [16]:
from scipy.stats import rankdata

# Train Ridge to predict the score RANK within each job, not the score itself
feat_rank = feat.copy()
feat_rank["rank_within_job"] = feat_rank.groupby("job_id")["matched_score"].transform(
    lambda s: rankdata(s) / len(s)
)

rows = []
for rf, jf, tr, te in iter_block_splits(feat_rank["resume_fold"], feat_rank["job_fold"]):
    est = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("model", Ridge(alpha=1.0, random_state=42)),
    ])
    est.fit(feat_rank.loc[tr, best_three], feat_rank.loc[tr, "rank_within_job"])
    y_pred = est.predict(feat_rank.loc[te, best_three])
    rows.append(within_job_spearman(feat_rank.loc[te, "matched_score"], y_pred, feat_rank.loc[te, "job_id"]))

print(f"Ridge trained on RANK   spearman={pd.Series(rows).mean():.3f}")

Ridge trained on RANK   spearman=0.366


In [17]:
ridge_rank_df = pd.Series(rows, index=pd.MultiIndex.from_tuples(
    [(rf, jf) for rf, jf, _, _ in iter_block_splits(feat_rank["resume_fold"], feat_rank["job_fold"])]
))
paired2 = baseline3.reset_index(drop=True).to_frame("baseline").join(
    ridge_rank_df.reset_index(drop=True).to_frame("ridge_rank")
)
from scipy.stats import wilcoxon
stat, p = wilcoxon(paired2["baseline"], paired2["ridge_rank"])
print(f"baseline gagne sur {(paired2['baseline'] > paired2['ridge_rank']).sum()}/25  p={p:.3f}")

for model_name in ["random_forest", "gradient_boosting"]:
    res = evaluate_config(feat, best_three, model_name)
    print(f"{model_name:20} spearman={res['within_job_spearman'].mean():.3f}  std={res['within_job_spearman'].std():.3f}")

baseline gagne sur 16/25  p=0.120
random_forest        spearman=0.261  std=0.076
gradient_boosting    spearman=0.287  std=0.072
